# 05 — SHAP Explainability & Fairness Audit

**Goal:** Make every risk decision explainable and fair.

This notebook fulfils two regulatory requirements:
1. **Right to Explanation** (GDPR Article 22): Show the top factors that
   drove each individual decision using SHAP.
2. **Fair Lending**: Audit demographic parity across gender and age groups
   to detect and document disparate impact.

## Contents
1. Load model and test data
2. Global SHAP feature importance (beeswarm + bar)
3. Individual explanations: One GREEN, YELLOW, RED applicant
4. SHAP dependence plots for top features
5. Fairness audit: gender and age band parity
6. PSI drift monitoring on a synthetic reference set

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from models import DefaultClassifier
from utils.config import load_config
from utils.fairness_audit import run_fairness_audit, add_age_band
from training.evaluate import compute_psi

sns.set_theme(style='whitegrid')
cfg = load_config()

clf = DefaultClassifier.load('../outputs/models', cfg)
print('Model loaded.')

## 1. Load Test Data

In [ ]:
master = pd.read_csv('../data/processed/feature_cache/master_features.csv')
obj_cols = master.select_dtypes('object').columns.tolist()
high_missing = master.columns[master.isnull().mean() > 0.5].tolist()
X_raw = master.drop(columns=['TARGET', 'SK_ID_CURR'] + obj_cols + high_missing, errors='ignore')
X = X_raw.reindex(columns=clf.feature_columns, fill_value=0).fillna(X_raw.median(numeric_only=True))
y = master['TARGET']

# Sample for SHAP (computationally expensive on full dataset)
sample_size = 2000
X_sample = X.sample(sample_size, random_state=42)
print(f'SHAP sample size: {len(X_sample)}')

## 2. Global SHAP Feature Importance

In [ ]:
# Use base XGBoost model (not calibrated wrapper) for SHAP
base_model = clf.model if clf.model else clf.calibrated_model.estimator
explainer = shap.TreeExplainer(base_model)
shap_values = explainer.shap_values(X_sample)

# Bar plot: mean |SHAP| per feature
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, plot_type='bar', max_display=20, show=False)
plt.title('Global SHAP Feature Importance (Mean |SHAP| value)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/05_shap_global_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm: shows direction of impact
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/05_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Individual Applicant Explanations

One RED, one YELLOW, and one GREEN applicant explained.

In [ ]:
proba = clf.predict_proba(X)
tl    = clf.predict_traffic_light(X)

# Pick representative applicants
explanations_cfg = [
    ('RED',    tl[tl['risk_band']=='RED'].index[0]),
    ('YELLOW', tl[tl['risk_band']=='YELLOW'].index[0]),
    ('GREEN',  tl[tl['risk_band']=='GREEN'].index[0]),
]

shap_expl = explainer(X)

for band, idx in explanations_cfg:
    row_pos = X.index.get_loc(idx)
    p = proba[row_pos]
    print(f'\n--- {band} applicant (index={idx}, risk_score={p:.3f}) ---')
    plt.figure(figsize=(10, 4))
    shap.plots.waterfall(shap_expl[row_pos], max_display=12, show=False)
    plt.title(f'{band} Applicant — SHAP Waterfall (P(default)={p:.1%})', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../outputs/figures/05_shap_waterfall_{band.lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## 4. Fairness Audit — Gender & Age Band Parity

In [ ]:
app_raw = pd.read_csv(cfg.data.application_train, usecols=['SK_ID_CURR','TARGET','CODE_GENDER','DAYS_BIRTH'])
app_raw = add_age_band(app_raw)
app_raw['predicted_band'] = tl['risk_band'].reindex(range(len(app_raw))).values

fairness_report = run_fairness_audit(
    app_raw,
    label_col='TARGET',
    protected_cols=['CODE_GENDER', 'age_band'],
    parity_tolerance=cfg.fairness.parity_tolerance,
)

import os
os.makedirs('../outputs/reports', exist_ok=True)
fairness_report.to_csv('../outputs/reports/fairness_report.csv', index=False)
print('Fairness report saved.')
fairness_report

## 5. PSI Drift Monitoring

In [ ]:
# Simulate a 'reference' distribution (first 60%) and 'current' (last 20%)
n = len(proba)
ref_scores = proba[:int(n * 0.6)]
cur_scores = proba[int(n * 0.8):]

psi_value = compute_psi(ref_scores, cur_scores, n_bins=10)
print(f'PSI value: {psi_value:.4f}')
if psi_value < 0.1:
    print('Status: ✅ STABLE — No significant score distribution shift')
elif psi_value < 0.25:
    print('Status: ⚠️  WARNING — Moderate shift detected, monitor closely')
else:
    print('Status: 🔴 CRITICAL — Significant shift, consider retraining')

# Visualise
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ref_scores, bins=20, alpha=0.6, color='#3498db', density=True, label='Reference (train)')
ax.hist(cur_scores, bins=20, alpha=0.6, color='#e74c3c', density=True, label='Current (test)')
ax.set_xlabel('Predicted default probability')
ax.set_title(f'Score Distribution: Reference vs Current | PSI={psi_value:.3f}', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/05_psi_drift.png', dpi=150, bbox_inches='tight')
plt.show()